<a href="https://colab.research.google.com/github/Numanur/heart-failure-monitoring-llm-rag/blob/main/Thesis_3_corpus_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install -q pymupdf pymupdf4llm pandas tqdm

In [27]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/llm")

RAW_DIR = BASE_DIR / "rag_corpus_raw"
PARSED_DIR = BASE_DIR / "rag_corpus_parsed"
META_DIR = BASE_DIR / "rag_metadata"

for d in [RAW_DIR, PARSED_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR   :", BASE_DIR)
print("RAW_DIR    :", RAW_DIR)
print("PARSED_DIR :", PARSED_DIR)
print("META_DIR   :", META_DIR)

BASE_DIR   : /content/drive/MyDrive/llm
RAW_DIR    : /content/drive/MyDrive/llm/rag_corpus_raw
PARSED_DIR : /content/drive/MyDrive/llm/rag_corpus_parsed
META_DIR   : /content/drive/MyDrive/llm/rag_metadata


In [29]:
import re
import json
import hashlib
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    import fitz  # PyMuPDF
except Exception as e:
    raise RuntimeError("PyMuPDF import failed. Re-run the install cell.") from e

try:
    import pymupdf4llm
    HAS_PYMUPDF4LLM = True
except Exception:
    HAS_PYMUPDF4LLM = False

print("pymupdf4llm available:", HAS_PYMUPDF4LLM)

pymupdf4llm available: True


In [30]:
DOCUMENTS = [
    {
        "doc_id": "aha_2022_guideline",
        "canonical_file": "1_aha_acc_hfsa_2022.pdf",
        "filename_keywords": ["aha", "acc", "hfsa", "guideline"],
        "fallback_keywords": ["heidenreich", "heart-failure-a-report"],
        "source_title": "2022 AHA/ACC/HFSA Guideline for the Management of Heart Failure",
        "publisher": "AHA/ACC/HFSA",
        "year": "2022",
        "last_updated": "2022",
        "document_type": "clinical_guideline",
        "audience": "clinician",
        "clinical_phase": "chronic, acute, post_discharge",
        "authority_level": "high",
        "parsed_md_file": "1_aha_acc_hfsa_2022.md",
        "parent_strategy": "aha_guideline_section_hierarchy",
        "child_strategy": "recommendation_synopsis_supportive_text_semantic_children",
        "min_page_for_headings": 4,
    },
    {
        "doc_id": "nice_chronic_hf",
        "canonical_file": "2_nice_chronic_hf.pdf",
        "filename_keywords": ["chronic", "heart", "failure"],
        "fallback_keywords": ["ng106", "chronic-heart-failure"],
        "source_title": "NICE Chronic heart failure in adults: diagnosis and management",
        "publisher": "NICE",
        "year": "2025",
        "last_updated": "2025-09-03",
        "document_type": "clinical_guideline",
        "audience": "clinician",
        "clinical_phase": "chronic, post_discharge",
        "authority_level": "high",
        "parsed_md_file": "2_nice_chronic_hf.md",
        "parent_strategy": "nice_numbered_section_hierarchy",
        "child_strategy": "individual_numbered_recommendations_plus_semantic_children",
        "min_page_for_headings": 6,
    },
    {
        "doc_id": "nice_acute_hf",
        "canonical_file": "3_nice_acute_hf.pdf",
        "filename_keywords": ["acute", "heart", "failure"],
        "fallback_keywords": ["cg187", "acute-heart-failure"],
        "source_title": "NICE Acute heart failure: diagnosis and management",
        "publisher": "NICE",
        "year": "2025",
        "last_updated": "2021-11-17",
        "document_type": "clinical_guideline",
        "audience": "clinician",
        "clinical_phase": "acute, post_discharge",
        "authority_level": "high",
        "parsed_md_file": "3_nice_acute_hf.md",
        "parent_strategy": "nice_numbered_section_hierarchy",
        "child_strategy": "individual_numbered_recommendations_plus_semantic_children",
        "min_page_for_headings": 7,
    },
    {
        "doc_id": "aha_discharge_packet",
        "canonical_file": "4_aha_discharge_packet.pdf",
        "filename_keywords": ["discharge", "packet"],
        "fallback_keywords": ["ds18660", "patient-discharge"],
        "source_title": "AHA Discharge Packet for Patients Diagnosed with Heart Failure",
        "publisher": "American Heart Association",
        "year": "2022",
        "last_updated": "2022",
        "document_type": "patient_education",
        "audience": "patient",
        "clinical_phase": "post_discharge, self_management",
        "authority_level": "supportive",
        "parsed_md_file": "4_aha_discharge_packet.md",
        "parent_strategy": "patient_education_page_topic_hierarchy",
        "child_strategy": "warning_signs_self_care_instructions_semantic_children",
        "min_page_for_headings": 3,
    },
]

doc_by_id = {d["doc_id"]: d for d in DOCUMENTS}

pd.DataFrame(DOCUMENTS)[[
    "doc_id",
    "source_title",
    "publisher",
    "document_type",
    "audience",
    "clinical_phase",
    "authority_level",
]]

,doc_id,source_title,publisher,document_type,audience,clinical_phase,authority_level
0,aha_2022_guideline,2022 AHA/ACC/HFSA Guideline for the Management...,AHA/ACC/HFSA,clinical_guideline,clinician,"chronic, acute, post_discharge",high
1,nice_chronic_hf,NICE Chronic heart failure in adults: diagnosi...,NICE,clinical_guideline,clinician,"chronic, post_discharge",high
2,nice_acute_hf,NICE Acute heart failure: diagnosis and manage...,NICE,clinical_guideline,clinician,"acute, post_discharge",high
3,aha_discharge_packet,AHA Discharge Packet for Patients Diagnosed wi...,American Heart Association,patient_education,patient,"post_discharge, self_management",supportive


In [31]:
def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()


def resolve_pdf_path(doc_cfg: Dict[str, Any], raw_dir: Path) -> Optional[Path]:
    canonical = raw_dir / doc_cfg["canonical_file"]
    if canonical.exists():
        return canonical

    pdfs = sorted(raw_dir.glob("*.pdf"))
    if not pdfs:
        return None

    keyword_sets = [
        doc_cfg.get("filename_keywords", []),
        doc_cfg.get("fallback_keywords", []),
    ]

    for keywords in keyword_sets:
        keywords_norm = [k.lower() for k in keywords]
        for p in pdfs:
            normalized = normalize_name(p.name)
            if all(k in normalized for k in keywords_norm):
                return p

    expected_prefix = doc_cfg["canonical_file"].split("_")[0]
    for p in pdfs:
        if p.name.strip().startswith(expected_prefix):
            return p

    return None


resolved_paths = {}
missing = []

for doc in DOCUMENTS:
    path = resolve_pdf_path(doc, RAW_DIR)
    resolved_paths[doc["doc_id"]] = path
    if path is None:
        missing.append(doc["doc_id"])

print("Resolved PDF paths:")
for doc_id, path in resolved_paths.items():
    print(f"- {doc_id}: {path}")

if missing:
    print("\nMissing documents:")
    for doc_id in missing:
        print(" -", doc_id)

    print("\nPlace the PDFs in:", RAW_DIR)
    raise FileNotFoundError("One or more required PDFs were not found in rag_corpus_raw.")

Resolved PDF paths:
- aha_2022_guideline: /content/drive/MyDrive/llm/rag_corpus_raw/1 heidenreich-et-al-2022-2022-aha-acc-hfsa-guideline-for-the-management-of-heart-failure-a-report-of-the-american-college.pdf
- nice_chronic_hf: /content/drive/MyDrive/llm/rag_corpus_raw/2 chronic-heart-failure-in-adults-diagnosis-and-management-pdf-66141541311685.pdf
- nice_acute_hf: /content/drive/MyDrive/llm/rag_corpus_raw/3 acute-heart-failure-diagnosis-and-management-pdf-35109817738693.pdf
- aha_discharge_packet: /content/drive/MyDrive/llm/rag_corpus_raw/4 DS18660_ENG_Patient-Discharge-Packet_2022.pdf


In [32]:
def stable_hash(text: str, length: int = 12) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:length]


def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def safe_strip(line: str) -> str:
    return re.sub(r"\s+", " ", line).strip()


FOOTER_PATTERNS = [
    r"^Downloaded from .*",
    r"^Circulation\. 2022;.*",
    r"^CLINICAL STATEMENTS\s*$",
    r"^AND GUIDELINES\s*$",
    r"^Heidenreich et al .*Heart Failure Guideline\s*$",
    r"^AHA Scientific Statements\s*$",
    r"^© NICE .*",
    r"^Page \d+ of\s*\d+\s*$",
    r"^Chronic heart failure in adults: diagnosis and management \(NG106\).*$",
    r"^Acute heart failure: diagnosis and management \(CG187\).*$",
    r"^www\.nice\.org\.uk/guidance/.*$",
    r"^We have many other fact sheets.*$",
    r"^Visit heart\.org/answersbyheart.*$",
    r"^HOW CAN I LEARN MORE\??$",
]


def remove_noise_lines(text: str) -> str:
    cleaned_lines = []

    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            cleaned_lines.append("")
            continue

        drop = False
        for pat in FOOTER_PATTERNS:
            if re.search(pat, line, flags=re.IGNORECASE):
                drop = True
                break

        if drop:
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def normalize_pdf_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\u00ad", "")
    text = text.replace("￾", "")
    text = text.replace(" ", " ")
    text = text.replace("•\u00a0", "• ")
    text = text.replace("\xa0", " ")

    text = remove_noise_lines(text)

    # Join words split by PDF line wrapping, e.g., recom-\nmendation.
    text = re.sub(r"([A-Za-z])-\n([A-Za-z])", r"\1\2", text)

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)

    return text.strip()


def compact_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

In [33]:
def parse_with_pymupdf4llm(pdf_path: Path) -> Optional[List[Dict[str, Any]]]:
    if not HAS_PYMUPDF4LLM:
        return None

    try:
        result = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)

        if isinstance(result, list):
            pages = []

            for i, item in enumerate(result):
                page_text = item.get("text", "") if isinstance(item, dict) else str(item)

                page_no = None
                if isinstance(item, dict):
                    meta = item.get("metadata", {}) or {}
                    page_no = meta.get("page") or meta.get("page_number")

                if page_no is None or int(page_no) <= 0:
                    page_no = i + 1

                pages.append({
                    "page": int(page_no),
                    "text": normalize_pdf_text(page_text),
                    "parser": "pymupdf4llm_page_chunks",
                })

            return pages

    except Exception as e:
        print(f"pymupdf4llm page_chunks failed for {pdf_path.name}: {e}")

    return None


def parse_with_pymupdf(pdf_path: Path) -> List[Dict[str, Any]]:
    pages = []
    doc = fitz.open(str(pdf_path))

    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page": i + 1,
            "text": normalize_pdf_text(text),
            "parser": "pymupdf_text",
        })

    doc.close()
    return pages


def parse_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    pages = parse_with_pymupdf4llm(pdf_path)

    if pages is None:
        pages = parse_with_pymupdf(pdf_path)

    return pages


def save_pages_as_markdown(doc_cfg: Dict[str, Any], pages: List[Dict[str, Any]]) -> Path:
    out_path = PARSED_DIR / doc_cfg["parsed_md_file"]

    with out_path.open("w", encoding="utf-8") as f:
        f.write(f"# {doc_cfg['source_title']}\n\n")
        f.write(f"- Document ID: `{doc_cfg['doc_id']}`\n")
        f.write(f"- Publisher: {doc_cfg['publisher']}\n")
        f.write(f"- Document type: {doc_cfg['document_type']}\n")
        f.write(f"- Authority level: {doc_cfg['authority_level']}\n\n")

        for page in pages:
            f.write("\n\n---\n\n")
            f.write(f"<!-- page: {page['page']} -->\n\n")
            f.write(page["text"].strip())
            f.write("\n")

    return out_path


all_parsed_pages = []

for doc_cfg in DOCUMENTS:
    doc_id = doc_cfg["doc_id"]
    pdf_path = resolved_paths[doc_id]

    print(f"\nParsing: {doc_id}")
    pages = parse_pdf_pages(pdf_path)

    md_path = save_pages_as_markdown(doc_cfg, pages)

    for p in pages:
        all_parsed_pages.append({
            "doc_id": doc_id,
            "source_file": pdf_path.name,
            "source_title": doc_cfg["source_title"],
            "publisher": doc_cfg["publisher"],
            "year": doc_cfg["year"],
            "last_updated": doc_cfg["last_updated"],
            "document_type": doc_cfg["document_type"],
            "audience": doc_cfg["audience"],
            "clinical_phase": doc_cfg["clinical_phase"],
            "authority_level": doc_cfg["authority_level"],
            "page": p["page"],
            "parser": p["parser"],
            "text": p["text"],
            "char_count": len(p["text"]),
        })

    print(f"  pages parsed : {len(pages)}")
    print(f"  saved md     : {md_path}")

parsed_pages_path = META_DIR / "parsed_pages.jsonl"
write_jsonl(all_parsed_pages, parsed_pages_path)

print("\nSaved page-level parsed text:", parsed_pages_path)
print("Total parsed pages:", len(all_parsed_pages))


Parsing: aha_2022_guideline
=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [34]:
inventory_rows = []

for doc_cfg in DOCUMENTS:
    doc_id = doc_cfg["doc_id"]
    pdf_path = resolved_paths[doc_id]
    pages = [p for p in all_parsed_pages if p["doc_id"] == doc_id]

    inventory_rows.append({
        "doc_id": doc_id,
        "source_file": pdf_path.name,
        "canonical_file": doc_cfg["canonical_file"],
        "source_title": doc_cfg["source_title"],
        "publisher": doc_cfg["publisher"],
        "year": doc_cfg["year"],
        "last_updated": doc_cfg["last_updated"],
        "document_type": doc_cfg["document_type"],
        "audience": doc_cfg["audience"],
        "clinical_phase": doc_cfg["clinical_phase"],
        "authority_level": doc_cfg["authority_level"],
        "raw_pdf_path": str(pdf_path),
        "parsed_md_path": str(PARSED_DIR / doc_cfg["parsed_md_file"]),
        "page_count": len(pages),
        "total_chars": sum(p["char_count"] for p in pages),
        "parent_strategy": doc_cfg["parent_strategy"],
        "child_strategy": doc_cfg["child_strategy"],
    })

inventory_df = pd.DataFrame(inventory_rows)
inventory_path = META_DIR / "document_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

print("Saved document inventory:", inventory_path)
inventory_df

Saved document inventory: /content/drive/MyDrive/llm/rag_metadata/document_inventory.csv


,doc_id,source_file,canonical_file,source_title,publisher,year,last_updated,document_type,audience,clinical_phase,authority_level,raw_pdf_path,parsed_md_path,page_count,total_chars,parent_strategy,child_strategy
0,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,1_aha_acc_hfsa_2022.pdf,2022 AHA/ACC/HFSA Guideline for the Management...,AHA/ACC/HFSA,2022,2022,clinical_guideline,clinician,"chronic, acute, post_discharge",high,/content/drive/MyDrive/llm/rag_corpus_raw/1 he...,/content/drive/MyDrive/llm/rag_corpus_parsed/1...,138,865528,aha_guideline_section_hierarchy,recommendation_synopsis_supportive_text_semant...
1,nice_chronic_hf,2 chronic-heart-failure-in-adults-diagnosis-an...,2_nice_chronic_hf.pdf,NICE Chronic heart failure in adults: diagnosi...,NICE,2025,2025-09-03,clinical_guideline,clinician,"chronic, post_discharge",high,/content/drive/MyDrive/llm/rag_corpus_raw/2 ch...,/content/drive/MyDrive/llm/rag_corpus_parsed/2...,39,55794,nice_numbered_section_hierarchy,individual_numbered_recommendations_plus_seman...
2,nice_acute_hf,3 acute-heart-failure-diagnosis-and-management...,3_nice_acute_hf.pdf,NICE Acute heart failure: diagnosis and manage...,NICE,2025,2021-11-17,clinical_guideline,clinician,"acute, post_discharge",high,/content/drive/MyDrive/llm/rag_corpus_raw/3 ac...,/content/drive/MyDrive/llm/rag_corpus_parsed/3...,19,24780,nice_numbered_section_hierarchy,individual_numbered_recommendations_plus_seman...
3,aha_discharge_packet,4 DS18660_ENG_Patient-Discharge-Packet_2022.pdf,4_aha_discharge_packet.pdf,AHA Discharge Packet for Patients Diagnosed wi...,American Heart Association,2022,2022,patient_education,patient,"post_discharge, self_management",supportive,/content/drive/MyDrive/llm/rag_corpus_raw/4 DS...,/content/drive/MyDrive/llm/rag_corpus_parsed/4...,48,72835,patient_education_page_topic_hierarchy,warning_signs_self_care_instructions_semantic_...


In [35]:
# ============================================================
# FIXED CELL 9
# Improved parent chunk helpers
# Fixes:
# - Markdown heading artifacts from pymupdf4llm
# - NICE heading detection
# - AHA false parent heading detection
# - source_file metadata
# - front matter exclusion
# ============================================================

import re
import json
import hashlib
from typing import Dict, List, Any, Optional, Tuple


def stable_hash(text: str, length: int = 12) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:length]


def write_jsonl(records: List[Dict[str, Any]], path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def safe_strip(line: str) -> str:
    return re.sub(r"\s+", " ", str(line)).strip()


def strip_markdown_heading_artifacts(line: str) -> str:
    """
    pymupdf4llm often adds Markdown syntax like:
    # 1.1 Heading
    **1.1 Heading**
    This function normalizes those lines before heading detection.
    """
    line = str(line).strip()

    # Remove HTML comments if any.
    line = re.sub(r"<!--.*?-->", " ", line)

    # Remove markdown heading hashes.
    line = re.sub(r"^\s{0,3}#{1,6}\s*", "", line)

    # Remove markdown bold/italic wrappers.
    line = line.replace("**", "").replace("__", "")
    line = line.replace("*", "")

    # Remove leading bullets that can appear before headings.
    line = re.sub(r"^[•\-–]\s*", "", line)

    return safe_strip(line)


def normalize_text_for_chunking(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\u00ad", "")
    text = text.replace("￾", "")
    text = text.replace("\xa0", " ")
    text = text.replace(" ", " ")

    # Remove repeated footer/header noise.
    noise_patterns = [
        r"Downloaded from .*",
        r"Circulation\. 2022;.*",
        r"CLINICAL STATEMENTS\s*",
        r"AND GUIDELINES\s*",
        r"Heidenreich et al .*Heart Failure Guideline\s*",
        r"AHA Scientific Statements\s*",
        r"© NICE .*",
        r"Page \d+ of\s*\d+",
        r"Chronic heart failure in adults: diagnosis and management \(NG106\).*",
        r"Acute heart failure: diagnosis and management \(CG187\).*",
        r"www\.nice\.org\.uk/guidance/.*",
        r"We have many other fact sheets.*",
        r"Visit heart\.org/answersbyheart.*",
    ]

    for pat in noise_patterns:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    # Join hyphenated line breaks.
    text = re.sub(r"([A-Za-z])-\n([A-Za-z])", r"\1\2", text)

    # Normalize repeated whitespace but preserve paragraph breaks.
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def compact_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def line_records_for_doc(doc_id: str) -> List[Dict[str, Any]]:
    """
    Convert parsed page records into page-aware cleaned line records.
    """
    pages = [p for p in all_parsed_pages if p["doc_id"] == doc_id]
    records = []

    for p in pages:
        page_no = int(p["page"])
        page_text = normalize_text_for_chunking(p.get("text", ""))

        for line in page_text.splitlines():
            clean_line = strip_markdown_heading_artifacts(line)
            if clean_line:
                records.append({
                    "page": page_no,
                    "line": clean_line,
                })

    return records


def is_front_matter_section(section_id: str, section_title: str, page_start: int) -> bool:
    """
    Marks non-evidence front matter so it can be skipped during child creation.
    """
    title = (section_title or "").lower()
    sid = (section_id or "").lower()

    front_terms = [
        "front matter",
        "table of contents",
        "contents",
        "overview",
        "who is it for",
        "introduction",
        "your responsibility",
        "preamble",
        "abstract",
        "key words",
        "abbreviations",
        "associated guidelines",
        "references",
        "appendix",
        "author relationships",
        "reviewer relationships",
    ]

    if sid == "front_matter":
        return True

    if any(term in title for term in front_terms):
        return True

    # Early pages in guideline PDFs often contain front matter.
    if page_start <= 3 and any(term in title for term in ["contents", "overview", "introduction"]):
        return True

    return False


def clean_parent_text(lines: List[str]) -> str:
    text = "\n".join(lines)
    text = normalize_text_for_chunking(text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def create_parent_node(
    doc_cfg: Dict[str, Any],
    parent_index: int,
    section_id: str,
    section_title: str,
    page_start: int,
    page_end: int,
    text: str,
) -> Dict[str, Any]:

    doc_id = doc_cfg["doc_id"]
    source_file = resolved_paths[doc_id].name if doc_id in resolved_paths else doc_cfg.get("canonical_file", "")

    parent_id = (
        f"{doc_id}_parent_"
        f"{parent_index:04d}_"
        f"{stable_hash(section_id + section_title + text, 8)}"
    )

    front_matter = is_front_matter_section(section_id, section_title, page_start)

    return {
        "parent_id": parent_id,
        "doc_id": doc_id,
        "text": text,
        "metadata": {
            "doc_id": doc_id,
            "source_file": source_file,
            "source_title": doc_cfg["source_title"],
            "publisher": doc_cfg["publisher"],
            "year": doc_cfg["year"],
            "last_updated": doc_cfg["last_updated"],
            "document_type": doc_cfg["document_type"],
            "audience": doc_cfg["audience"],
            "clinical_phase": doc_cfg["clinical_phase"],
            "authority_level": doc_cfg["authority_level"],
            "section_id": section_id,
            "section_title": section_title,
            "page_start": int(page_start),
            "page_end": int(page_end),
            "parent_strategy": doc_cfg["parent_strategy"],
            "is_front_matter": front_matter,
            "index_exclude": front_matter,
        },
    }

In [36]:
# ============================================================
# FIXED CELL 10
# NICE parent chunking
# Fixes:
# - NICE chronic and acute no longer become one giant parent
# - Splits by true section headings: 1.1, 1.2, 1.3, etc.
# - Avoids table-of-contents headings by using min page threshold
# ============================================================

def detect_nice_parent_heading(line: str) -> Optional[Tuple[str, str]]:
    """
    Detect NICE parent headings such as:
    1.1 Team working in the management of heart failure
    1.7 Starting and monitoring medication use

    Does not treat recommendation IDs like 1.7.4 as parent headings.
    """
    line = strip_markdown_heading_artifacts(line)

    # Parent heading: 1.1 Title
    # Exclude child recommendation IDs: 1.1.1, 1.7.4, etc.
    m = re.match(r"^(1\.\d{1,2})\s+(.{3,180})$", line)

    if not m:
        return None

    section_id = m.group(1).strip()
    section_title = m.group(2).strip()

    # Exclude recommendation IDs.
    if re.match(r"^1\.\d{1,2}\.\d+", line):
        return None

    # Exclude table of contents page ranges like "1.1 Something .... 6"
    section_title = re.sub(r"\.{2,}\s*\d+\s*$", "", section_title).strip()

    bad_titles = [
        "recommendations",
        "recommendations for research",
        "rationale and impact",
        "finding more information",
        "update information",
        "terms used",
        "context",
    ]

    # Some of these are real sections, but not core evidence sections for this thesis.
    # We keep "Recommendations" itself out, but keep numbered clinical sections.
    if section_title.lower() in bad_titles:
        return None

    return section_id, section_title


def build_nice_section_parent_nodes(doc_cfg: Dict[str, Any]) -> List[Dict[str, Any]]:
    records = line_records_for_doc(doc_cfg["doc_id"])
    min_page_for_headings = int(doc_cfg.get("min_page_for_headings", 1))

    parents = []
    current_section_id = "front_matter"
    current_section_title = "Front matter / introductory material"
    current_lines = []
    current_page_start = None
    parent_index = 1

    for rec in records:
        page = int(rec["page"])
        line = rec["line"]

        heading = None
        if page >= min_page_for_headings:
            heading = detect_nice_parent_heading(line)

        if heading is not None:
            # Close previous parent.
            if current_lines:
                text = clean_parent_text(current_lines)
                if len(text) >= 150:
                    parents.append(create_parent_node(
                        doc_cfg=doc_cfg,
                        parent_index=parent_index,
                        section_id=current_section_id,
                        section_title=current_section_title,
                        page_start=current_page_start or page,
                        page_end=max(page - 1, current_page_start or page),
                        text=text,
                    ))
                    parent_index += 1

            current_section_id, current_section_title = heading
            current_lines = [line]
            current_page_start = page

        else:
            if current_page_start is None:
                current_page_start = page
            current_lines.append(line)

    # Close final parent.
    if current_lines:
        text = clean_parent_text(current_lines)
        if len(text) >= 150:
            last_page = int(records[-1]["page"]) if records else current_page_start
            parents.append(create_parent_node(
                doc_cfg=doc_cfg,
                parent_index=parent_index,
                section_id=current_section_id,
                section_title=current_section_title,
                page_start=current_page_start or 1,
                page_end=last_page,
                text=text,
            ))

    return parents

In [37]:
# ============================================================
# FIXED CELL 11
# AHA/ACC/HFSA parent chunking
# Fixes:
# - Requires real section IDs with dots, e.g. 4.1, 4.1.1, 7.3.8, 9.6
# - Prevents false parents like "1. Clinical congestion can be assessed..."
# ============================================================

def detect_aha_parent_heading(line: str) -> Optional[Tuple[str, str]]:
    """
    Detect true AHA guideline section headings.

    Accept:
    4.1 Clinical Assessment: History and Physical Examination
    4.1.1 Initial Laboratory and Electrocardiographic Testing
    4.2 Use of Biomarkers...
    7.1.1 Self-Care Support in HF
    9.6 Integration of Care...

    Reject:
    1. Clinical congestion can be assessed...
    2. Some patients with HF...
    Recommendation list numbering
    """
    line = strip_markdown_heading_artifacts(line)

    # Must contain at least one dot in the section ID.
    # This blocks false headings like "1. Clinical congestion..."
    m = re.match(
        r"^((?:[1-9]|1[0-4])\.\d+[a-z]?(?:\.\d+[a-z]?)*)(?:\.)?\s+(.{4,180})$",
        line
    )

    if not m:
        return None

    section_id = m.group(1).strip()
    section_title = m.group(2).strip()

    # Remove accidental trailing page references.
    section_title = re.sub(r"\.{2,}\s*e?\d+\s*$", "", section_title).strip()

    # Reject sentence-like false positives.
    bad_starts = [
        "In patients",
        "For patients",
        "Patients ",
        "Measurement ",
        "The use",
        "The ",
        "A ",
        "An ",
        "When ",
        "If ",
        "Some ",
        "Clinical congestion",
        "Laboratory evaluation",
        "Electrocardiography",
    ]

    if any(section_title.startswith(x) for x in bad_starts):
        return None

    # Reject overly long headings.
    if len(section_title) > 150:
        return None

    # Reject headings that look like references or table text.
    if re.search(r"\d+–\d+", section_title):
        return None

    return section_id, section_title


def build_aha_section_parent_nodes(doc_cfg: Dict[str, Any]) -> List[Dict[str, Any]]:
    records = line_records_for_doc(doc_cfg["doc_id"])
    min_page_for_headings = int(doc_cfg.get("min_page_for_headings", 1))

    parents = []
    current_section_id = "front_matter"
    current_section_title = "Front matter / introductory material"
    current_lines = []
    current_page_start = None
    parent_index = 1

    for rec in records:
        page = int(rec["page"])
        line = rec["line"]

        heading = None
        if page >= min_page_for_headings:
            heading = detect_aha_parent_heading(line)

        if heading is not None:
            # Close previous parent.
            if current_lines:
                text = clean_parent_text(current_lines)
                if len(text) >= 250:
                    parents.append(create_parent_node(
                        doc_cfg=doc_cfg,
                        parent_index=parent_index,
                        section_id=current_section_id,
                        section_title=current_section_title,
                        page_start=current_page_start or page,
                        page_end=max(page - 1, current_page_start or page),
                        text=text,
                    ))
                    parent_index += 1

            current_section_id, current_section_title = heading
            current_lines = [line]
            current_page_start = page

        else:
            if current_page_start is None:
                current_page_start = page
            current_lines.append(line)

    # Close final parent.
    if current_lines:
        text = clean_parent_text(current_lines)
        if len(text) >= 250:
            last_page = int(records[-1]["page"]) if records else current_page_start
            parents.append(create_parent_node(
                doc_cfg=doc_cfg,
                parent_index=parent_index,
                section_id=current_section_id,
                section_title=current_section_title,
                page_start=current_page_start or 1,
                page_end=last_page,
                text=text,
            ))

    return parents

In [38]:
# ============================================================
# FIXED CELL 12
# AHA Discharge Packet parent chunking
# Fixes:
# - Excludes table of contents page
# - Uses page/topic-level parent chunks
# - Adds source_file metadata through create_parent_node()
# ============================================================

DISCHARGE_TOPIC_KEYWORDS = [
    ("What is Heart Failure?", [
        "what is heart failure",
        "signs of heart failure",
        "causes of heart failure",
    ]),
    ("Types of Heart Failure", [
        "types of heart failure",
        "left-sided heart failure",
        "right-sided heart failure",
        "heart failure with congestion",
    ]),
    ("Ejection Fraction Explained", [
        "ejection fraction explained",
        "preserved ejection fraction",
        "mildly reduced ejection fraction",
        "reduced ejection fraction",
    ]),
    ("How Can I Live With Heart Failure?", [
        "how can i live",
        "what medicine might i take",
        "what should i watch out for",
    ]),
    ("Heart Failure Medications", [
        "heart failure medications",
        "ace inhibitors",
        "beta blockers",
        "diuretics",
        "sglt2",
    ]),
    ("Medication Adherence", [
        "take medications exactly as prescribed",
        "keep a list of all medications",
        "daily pill organizer",
        "do not stop taking",
    ]),
    ("Lifestyle Changes", [
        "lifestyle changes",
        "quitting smoking",
        "staying active",
        "alcohol intake",
    ]),
    ("Self-Check Plan", [
        "self-check plan",
        "medical alert",
        "pay attention",
        "daily weight check",
        "call your physician",
        "call 911",
    ]),
    ("Sodium and Diet", [
        "sodium",
        "diet",
        "nutrition",
        "hidden sources of sodium",
        "low-sodium",
        "salt",
    ]),
    ("Weight Management", [
        "weight management",
        "daily weight",
        "weight gain",
    ]),
    ("Charts and Logs", [
        "charts",
        "logs",
        "appointments",
        "keeping your appointments",
    ]),
]


def infer_discharge_topic(text: str, page: int) -> str:
    text_lower = text.lower()

    for title, keywords in DISCHARGE_TOPIC_KEYWORDS:
        if any(k.lower() in text_lower for k in keywords):
            return title

    for line in text.splitlines():
        line = safe_strip(line)
        if 5 <= len(line) <= 80 and not line.isdigit():
            return line

    return f"Patient education page {page}"


def is_discharge_front_page(text: str, page: int) -> bool:
    text_lower = text.lower()

    if page <= 3:
        if "table of contents" in text_lower or "discharge packet for patients" in text_lower:
            return True

    return False


def build_discharge_parent_nodes(doc_cfg: Dict[str, Any]) -> List[Dict[str, Any]]:
    pages = [p for p in all_parsed_pages if p["doc_id"] == doc_cfg["doc_id"]]

    parents = []
    parent_index = 1

    for p in pages:
        page_no = int(p["page"])
        text = normalize_text_for_chunking(p.get("text", ""))

        if len(text) < 120:
            continue

        if is_discharge_front_page(text, page_no):
            section_id = f"page_{page_no:03d}"
            topic = "Front matter / table of contents"

            parent = create_parent_node(
                doc_cfg=doc_cfg,
                parent_index=parent_index,
                section_id=section_id,
                section_title=topic,
                page_start=page_no,
                page_end=page_no,
                text=text,
            )
            parent["metadata"]["is_front_matter"] = True
            parent["metadata"]["index_exclude"] = True
            parents.append(parent)
            parent_index += 1
            continue

        topic = infer_discharge_topic(text, page_no)
        section_id = f"page_{page_no:03d}"

        parents.append(create_parent_node(
            doc_cfg=doc_cfg,
            parent_index=parent_index,
            section_id=section_id,
            section_title=topic,
            page_start=page_no,
            page_end=page_no,
            text=text,
        ))

        parent_index += 1

    return parents

In [39]:
# ============================================================
# FIXED CELL 13
# Rebuild all parent nodes using corrected parent splitters
# ============================================================

all_parent_nodes = []

for doc_cfg in DOCUMENTS:
    doc_id = doc_cfg["doc_id"]
    print(f"Building corrected parents for: {doc_id}")

    if doc_id == "aha_2022_guideline":
        parents = build_aha_section_parent_nodes(doc_cfg)

    elif doc_id in ["nice_chronic_hf", "nice_acute_hf"]:
        parents = build_nice_section_parent_nodes(doc_cfg)

    elif doc_id == "aha_discharge_packet":
        parents = build_discharge_parent_nodes(doc_cfg)

    else:
        raise ValueError(f"Unknown document id: {doc_id}")

    all_parent_nodes.extend(parents)
    print(f"  parents created: {len(parents)}")

parent_nodes_path = META_DIR / "parent_nodes.jsonl"
write_jsonl(all_parent_nodes, parent_nodes_path)

print("\nSaved corrected parent nodes:", parent_nodes_path)
print("Total corrected parent nodes:", len(all_parent_nodes))

parent_summary = pd.DataFrame([
    {
        "parent_id": p["parent_id"],
        "doc_id": p["doc_id"],
        "source_file": p["metadata"]["source_file"],
        "section_id": p["metadata"]["section_id"],
        "section_title": p["metadata"]["section_title"],
        "page_start": p["metadata"]["page_start"],
        "page_end": p["metadata"]["page_end"],
        "is_front_matter": p["metadata"]["is_front_matter"],
        "index_exclude": p["metadata"]["index_exclude"],
        "chars": len(p["text"]),
    }
    for p in all_parent_nodes
])

display(parent_summary.groupby("doc_id").agg(
    parent_count=("parent_id", "count"),
    index_excluded=("index_exclude", "sum"),
    total_chars=("chars", "sum"),
    avg_chars=("chars", "mean"),
).reset_index())

display(parent_summary.head(20))

Building corrected parents for: aha_2022_guideline
  parents created: 86
Building corrected parents for: nice_chronic_hf
  parents created: 13
Building corrected parents for: nice_acute_hf
  parents created: 8
Building corrected parents for: aha_discharge_packet
  parents created: 42

Saved corrected parent nodes: /content/drive/MyDrive/llm/rag_metadata/parent_nodes.jsonl
Total corrected parent nodes: 149


,doc_id,parent_count,index_excluded,total_chars,avg_chars
0,aha_2022_guideline,86,2,859223,9990.965116
1,aha_discharge_packet,42,1,72555,1727.500000
2,nice_acute_hf,8,1,24194,3024.250000
3,nice_chronic_hf,13,1,53729,4133.000000


,parent_id,doc_id,source_file,section_id,section_title,page_start,page_end,is_front_matter,index_exclude,chars
0,aha_2022_guideline_parent_0001_2f797e19,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,front_matter,Front matter / introductory material,1,4,True,True,22350
1,aha_2022_guideline_parent_0002_ca260a6a,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,1.2,Organization of the Writing Committee,5,5,False,False,1170
2,aha_2022_guideline_parent_0003_788883ec,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,1.1,Methodology and Evidence Review,5,5,False,False,1375
3,aha_2022_guideline_parent_0004_f075c0ed,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,1.4,Scope of the Guideline,5,5,False,False,2389
4,aha_2022_guideline_parent_0005_18c7e1cc,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,1.5,Class of Recommendation and Level of Evidence,6,6,False,False,421
5,aha_2022_guideline_parent_0006_4114404a,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,1.6,Abbreviations,6,6,True,True,3939
6,aha_2022_guideline_parent_0007_c83c5d84,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,2.1,Stages of HF,7,7,False,False,1846
7,aha_2022_guideline_parent_0008_d5b9e790,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,2.2,Classification of HF by Left Ventricular Eject...,7,10,False,False,13720
8,aha_2022_guideline_parent_0009_cff5d213,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,2.3,Diagnostic Algorithm for Classification of HF ...,11,11,False,False,4121
9,aha_2022_guideline_parent_0010_38a36ed2,aha_2022_guideline,1 heidenreich-et-al-2022-2022-aha-acc-hfsa-gui...,3.1,Epidemiology of HF,12,12,False,False,3340


In [40]:
# ============================================================
# FIXED CELL 14
# Child chunk helpers
# Fixes:
# - source_file metadata added to every child
# - index_exclude inherited from parent
# - better evidence role and topic metadata
# ============================================================

TOPIC_KEYWORDS = {
    "post_discharge_follow_up": [
        "discharge",
        "postdischarge",
        "post-discharge",
        "follow-up",
        "follow up",
        "within 2 weeks",
        "care plan",
        "transition",
        "transitions",
        "team-based",
        "primary care",
        "specialist heart failure team",
    ],
    "monitoring": [
        "monitor",
        "monitoring",
        "clinical review",
        "vital signs",
        "blood pressure",
        "heart rate",
        "weight",
        "urine output",
        "symptoms",
        "congestion",
        "physical examination",
    ],
    "renal_function_electrolytes": [
        "renal function",
        "kidney",
        "creatinine",
        "electrolyte",
        "electrolytes",
        "potassium",
        "egfr",
        "eGFR",
        "blood urea nitrogen",
        "urea",
    ],
    "natriuretic_peptides": [
        "BNP",
        "NT-proBNP",
        "natriuretic peptide",
        "natriuretic peptides",
    ],
    "medication_management": [
        "ACE inhibitor",
        "ACE inhibitors",
        "ACEi",
        "ARB",
        "ARNI",
        "beta-blocker",
        "beta blocker",
        "MRA",
        "mineralocorticoid",
        "SGLT2",
        "diuretic",
        "diuretics",
        "aldosterone antagonist",
        "digoxin",
        "medication",
        "medicines",
        "dose increment",
        "starting treatment",
        "treatment after stabilisation",
    ],
    "warning_signs": [
        "warning",
        "worsening",
        "shortness of breath",
        "dyspnea",
        "breath",
        "swelling",
        "edema",
        "weight gain",
        "chest pain",
        "dizziness",
        "confusion",
        "cough",
        "cannot lie flat",
        "fever",
        "medical alert",
        "call 911",
    ],
    "self_management": [
        "self-care",
        "self care",
        "daily weight",
        "low-sodium",
        "low sodium",
        "sodium",
        "salt",
        "diet",
        "exercise",
        "activity",
        "appointments",
        "adherence",
        "smoking",
        "alcohol",
    ],
    "diagnosis_assessment": [
        "diagnosis",
        "assessment",
        "echocardiography",
        "ECG",
        "chest x-ray",
        "history",
        "physical examination",
        "clinical examination",
        "blood tests",
    ],
}


def infer_topics(text: str) -> List[str]:
    topics = []
    text_lower = text.lower()

    for topic, keywords in TOPIC_KEYWORDS.items():
        for kw in keywords:
            if kw.lower() in text_lower:
                topics.append(topic)
                break

    return topics or ["general_heart_failure"]


def infer_evidence_role(
    text: str,
    doc_cfg: Dict[str, Any],
    section_title: str = "",
    marker_context: str = "",
) -> str:
    combined = f"{section_title} {marker_context} {text}".lower()
    text_stripped = text.strip()

    if doc_cfg["document_type"] == "patient_education":
        if any(k in combined for k in [
            "warning",
            "self-check",
            "medical alert",
            "call 911",
            "weight gain",
            "shortness of breath",
            "swelling",
        ]):
            return "warning_sign_or_self_check_instruction"

        if any(k in combined for k in [
            "medication",
            "take medications",
            "medicine",
            "prescribed",
        ]):
            return "patient_medication_instruction"

        if any(k in combined for k in [
            "sodium",
            "salt",
            "diet",
            "nutrition",
            "exercise",
            "activity",
            "smoking",
            "alcohol",
        ]):
            return "patient_lifestyle_instruction"

        return "patient_instruction"

    if re.match(r"^\s*1\.\d+\.\d+\b", text_stripped):
        return "recommendation"

    if "recommendation-specific supportive text" in combined:
        return "supportive_text"

    if "synopsis" in combined:
        return "synopsis"

    if any(k in combined for k in [
        "recommendations for",
        "cor loe recommendations",
        "recommendation",
    ]):
        return "recommendation"

    if "table" in combined:
        return "table_or_structured_evidence"

    return "supportive_text"


def split_into_sentences(text: str) -> List[str]:
    text = compact_whitespace(text)

    if not text:
        return []

    protected = text
    protected = protected.replace("e.g.", "eg")
    protected = protected.replace("i.e.", "ie")
    protected = protected.replace("U.S.", "US")

    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", protected)
    return [p.strip() for p in parts if p.strip()]


def semantic_child_split(
    text: str,
    min_chars: int = 250,
    target_chars: int = 900,
    max_chars: int = 1400,
) -> List[str]:
    """
    Structure-preserving semantic splitting.

    This is not fixed-size chunking.
    It respects paragraphs, bullets, and sentence boundaries.
    Character limits are only safeguards.
    """
    text = normalize_text_for_chunking(text)

    if not text:
        return []

    if len(text) < min_chars:
        return [text]

    raw_parts = re.split(r"\n\s*\n|(?=\n?\s*[•\-–]\s+)", text)
    parts = []

    for part in raw_parts:
        part = part.strip()
        if not part:
            continue

        if len(part) <= max_chars:
            parts.append(part)
        else:
            sentences = split_into_sentences(part)
            buf = ""

            for s in sentences:
                if not buf:
                    buf = s
                elif len(buf) + 1 + len(s) <= target_chars:
                    buf += " " + s
                else:
                    if buf:
                        parts.append(buf.strip())
                    buf = s

            if buf:
                parts.append(buf.strip())

    chunks = []
    buf = ""

    for part in parts:
        if not buf:
            buf = part
        elif len(buf) + 2 + len(part) <= max_chars:
            buf += "\n" + part
        else:
            if len(buf) < min_chars and chunks:
                chunks[-1] = chunks[-1] + "\n" + buf
            else:
                chunks.append(buf.strip())
            buf = part

    if buf:
        if len(buf) < min_chars and chunks:
            chunks[-1] = chunks[-1] + "\n" + buf
        else:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c.strip()) >= 80]


def extract_recommendation_id(text: str) -> Optional[str]:
    m = re.match(r"^\s*(1\.\d+\.\d+)\b", text.strip())
    if m:
        return m.group(1)
    return None


def make_child_node(
    doc_cfg: Dict[str, Any],
    parent: Dict[str, Any],
    child_index: int,
    text: str,
    evidence_role: str,
    recommendation_id: Optional[str] = None,
    marker_context: str = "",
    page_start: Optional[int] = None,
    page_end: Optional[int] = None,
) -> Dict[str, Any]:

    parent_meta = parent["metadata"]

    node_id = (
        f"{doc_cfg['doc_id']}_child_"
        f"{child_index:05d}_"
        f"{stable_hash(parent['parent_id'] + text, 8)}"
    )

    use_page_start = int(page_start) if page_start is not None else int(parent_meta.get("page_start"))
    use_page_end = int(page_end) if page_end is not None else int(parent_meta.get("page_end"))

    metadata = {
        "node_id": node_id,
        "parent_id": parent["parent_id"],
        "doc_id": doc_cfg["doc_id"],
        "source_file": parent_meta.get("source_file"),
        "source_title": doc_cfg["source_title"],
        "publisher": doc_cfg["publisher"],
        "year": doc_cfg["year"],
        "last_updated": doc_cfg["last_updated"],
        "document_type": doc_cfg["document_type"],
        "audience": doc_cfg["audience"],
        "clinical_phase": doc_cfg["clinical_phase"],
        "authority_level": doc_cfg["authority_level"],
        "section_id": parent_meta.get("section_id"),
        "section_title": parent_meta.get("section_title"),
        "recommendation_id": recommendation_id,
        "evidence_role": evidence_role,
        "topic": infer_topics(text),
        "page_start": use_page_start,
        "page_end": use_page_end,
        "parent_strategy": doc_cfg["parent_strategy"],
        "child_strategy": doc_cfg["child_strategy"],
        "marker_context": marker_context,
        "is_front_matter": bool(parent_meta.get("is_front_matter", False)),
        "index_exclude": bool(parent_meta.get("index_exclude", False)),
        "char_count": len(text),
    }

    return {
        "node_id": node_id,
        "parent_id": parent["parent_id"],
        "doc_id": doc_cfg["doc_id"],
        "text": text.strip(),
        "metadata": metadata,
    }

In [41]:
# ============================================================
# FIXED CELL 15
# NICE child chunking
# Fixes:
# - Uses corrected NICE parent sections
# - Extracts individual recommendation IDs
# - Inherits better section-level page ranges
# ============================================================

def split_nice_numbered_recommendations(text: str) -> List[Tuple[str, str]]:
    pattern = re.compile(r"(?m)^\s*(1\.\d+\.\d+)\b\s*")
    matches = list(pattern.finditer(text))

    recs = []

    for i, m in enumerate(matches):
        rec_id = m.group(1)
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        rec_text = text[start:end].strip()
        rec_text = normalize_text_for_chunking(rec_text)

        # Stop before rationale boilerplate if it got merged.
        rec_text = re.split(
            r"\n?For a short explanation of why the committee made|\n?Full details of the evidence",
            rec_text,
            maxsplit=1,
            flags=re.IGNORECASE,
        )[0].strip()

        if len(rec_text) >= 50:
            recs.append((rec_id, rec_text))

    return recs


def build_nice_child_nodes(
    doc_cfg: Dict[str, Any],
    parents: List[Dict[str, Any]],
    start_index: int = 1,
) -> List[Dict[str, Any]]:

    children = []
    child_index = start_index

    for parent in parents:
        if parent["metadata"].get("index_exclude", False):
            continue

        text = parent["text"]
        recs = split_nice_numbered_recommendations(text)

        if recs:
            for rec_id, rec_text in recs:
                children.append(make_child_node(
                    doc_cfg=doc_cfg,
                    parent=parent,
                    child_index=child_index,
                    text=rec_text,
                    evidence_role="recommendation",
                    recommendation_id=rec_id,
                    marker_context="NICE numbered recommendation",
                ))
                child_index += 1

        else:
            for chunk in semantic_child_split(text):
                role = infer_evidence_role(
                    chunk,
                    doc_cfg,
                    parent["metadata"].get("section_title", ""),
                )

                children.append(make_child_node(
                    doc_cfg=doc_cfg,
                    parent=parent,
                    child_index=child_index,
                    text=chunk,
                    evidence_role=role,
                    recommendation_id=extract_recommendation_id(chunk),
                    marker_context="NICE semantic supportive chunk",
                ))
                child_index += 1

    return children

In [42]:
# ============================================================
# FIXED CELL 16
# AHA guideline child chunking
# Fixes:
# - Skips front matter
# - Uses corrected AHA parent sections
# - Keeps recommendation/synopsis/supportive-text structure
# ============================================================

def split_aha_by_markers(text: str) -> List[Tuple[str, str]]:
    marker_regex = (
        r"(?im)^\s*("
        r"Recommendations? for .+|"
        r"Synopsis|"
        r"Recommendation-Specific Supportive Text|"
        r"Value Statement.*|"
        r"Table \d+\..*|"
        r"Figure \d+\..*"
        r")\s*$"
    )

    matches = list(re.finditer(marker_regex, text))

    if not matches:
        return [("section_text", text)]

    blocks = []

    if matches[0].start() > 0:
        lead = text[:matches[0].start()].strip()
        if len(lead) >= 150:
            blocks.append(("section_intro", lead))

    for i, m in enumerate(matches):
        marker = safe_strip(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        block = text[start:end].strip()

        if len(block) >= 120:
            blocks.append((marker, block))

    return blocks


def maybe_split_aha_recommendation_block(block_text: str) -> List[str]:
    pattern = re.compile(
        r"(?m)(?:^|\n)\s*(\d+)\.\s+"
        r"(In patients|For patients|In selected|For all|Patients|Measurement|The use|A follow-up)",
        re.IGNORECASE,
    )

    matches = list(pattern.finditer(block_text))

    if len(matches) >= 2:
        chunks = []

        for i, m in enumerate(matches):
            start = m.start()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(block_text)

            chunk = block_text[start:end].strip()

            if len(chunk) >= 80:
                chunks.append(chunk)

        return chunks

    return semantic_child_split(block_text)


def build_aha_child_nodes(
    doc_cfg: Dict[str, Any],
    parents: List[Dict[str, Any]],
    start_index: int = 1,
) -> List[Dict[str, Any]]:

    children = []
    child_index = start_index

    for parent in parents:
        if parent["metadata"].get("index_exclude", False):
            continue

        section_title = parent["metadata"].get("section_title", "")
        section_id = parent["metadata"].get("section_id", "")

        blocks = split_aha_by_markers(parent["text"])

        for marker, block_text in blocks:
            role = infer_evidence_role(
                block_text,
                doc_cfg,
                section_title=section_title,
                marker_context=marker,
            )

            if "recommendation" in role.lower():
                subchunks = maybe_split_aha_recommendation_block(block_text)
            else:
                subchunks = semantic_child_split(block_text)

            for j, chunk in enumerate(subchunks, start=1):
                rec_id = None

                if "recommendation" in role.lower():
                    rec_id = f"{section_id}_rec_{j:02d}" if section_id else None

                children.append(make_child_node(
                    doc_cfg=doc_cfg,
                    parent=parent,
                    child_index=child_index,
                    text=chunk,
                    evidence_role=role,
                    recommendation_id=rec_id,
                    marker_context=marker,
                ))

                child_index += 1

    return children

In [43]:
# ============================================================
# FIXED CELL 17
# AHA discharge packet child chunking
# Fixes:
# - Skips cover/table-of-contents chunks
# - Keeps warning signs and self-care instructions
# ============================================================

def split_patient_education_items(text: str) -> List[str]:
    text = normalize_text_for_chunking(text)

    chunks = semantic_child_split(
        text,
        min_chars=180,
        target_chars=700,
        max_chars=1100,
    )

    return chunks


def build_discharge_child_nodes(
    doc_cfg: Dict[str, Any],
    parents: List[Dict[str, Any]],
    start_index: int = 1,
) -> List[Dict[str, Any]]:

    children = []
    child_index = start_index

    for parent in parents:
        if parent["metadata"].get("index_exclude", False):
            continue

        section_title = parent["metadata"].get("section_title", "")
        chunks = split_patient_education_items(parent["text"])

        for chunk in chunks:
            role = infer_evidence_role(
                chunk,
                doc_cfg,
                section_title=section_title,
                marker_context="AHA patient discharge education",
            )

            children.append(make_child_node(
                doc_cfg=doc_cfg,
                parent=parent,
                child_index=child_index,
                text=chunk,
                evidence_role=role,
                recommendation_id=None,
                marker_context="AHA discharge packet topic/page child",
            ))

            child_index += 1

    return children

In [44]:
# ============================================================
# FIXED CELL 18
# Rebuild all child evidence nodes
# ============================================================

parents_by_doc = {}

for p in all_parent_nodes:
    parents_by_doc.setdefault(p["doc_id"], []).append(p)

all_child_nodes = []
child_start = 1

for doc_cfg in DOCUMENTS:
    doc_id = doc_cfg["doc_id"]
    parents = parents_by_doc.get(doc_id, [])

    print(f"Building corrected child evidence nodes for: {doc_id}")

    if doc_id == "aha_2022_guideline":
        children = build_aha_child_nodes(
            doc_cfg,
            parents,
            start_index=child_start,
        )

    elif doc_id in ["nice_chronic_hf", "nice_acute_hf"]:
        children = build_nice_child_nodes(
            doc_cfg,
            parents,
            start_index=child_start,
        )

    elif doc_id == "aha_discharge_packet":
        children = build_discharge_child_nodes(
            doc_cfg,
            parents,
            start_index=child_start,
        )

    else:
        raise ValueError(f"Unknown document id: {doc_id}")

    all_child_nodes.extend(children)
    child_start += len(children)

    print(f"  children created: {len(children)}")

child_nodes_path = META_DIR / "child_nodes.jsonl"
write_jsonl(all_child_nodes, child_nodes_path)

print("\nSaved corrected child nodes:", child_nodes_path)
print("Total corrected child evidence nodes:", len(all_child_nodes))

Building corrected child evidence nodes for: aha_2022_guideline
  children created: 890
Building corrected child evidence nodes for: nice_chronic_hf
  children created: 87
Building corrected child evidence nodes for: nice_acute_hf
  children created: 28
Building corrected child evidence nodes for: aha_discharge_packet
  children created: 84

Saved corrected child nodes: /content/drive/MyDrive/llm/rag_metadata/child_nodes.jsonl
Total corrected child evidence nodes: 1089


In [45]:
# ============================================================
# FIXED CELL 19
# Corpus quality summary after fixes
# ============================================================

child_rows = []

for c in all_child_nodes:
    m = c["metadata"]

    child_rows.append({
        "node_id": c["node_id"],
        "doc_id": c["doc_id"],
        "source_file": m["source_file"],
        "source_title": m["source_title"],
        "document_type": m["document_type"],
        "audience": m["audience"],
        "authority_level": m["authority_level"],
        "section_id": m["section_id"],
        "section_title": m["section_title"],
        "recommendation_id": m["recommendation_id"],
        "evidence_role": m["evidence_role"],
        "topic": ", ".join(m["topic"]),
        "page_start": m["page_start"],
        "page_end": m["page_end"],
        "is_front_matter": m["is_front_matter"],
        "index_exclude": m["index_exclude"],
        "char_count": m["char_count"],
    })

child_df = pd.DataFrame(child_rows)

summary_df = child_df.groupby([
    "doc_id",
    "document_type",
    "authority_level",
]).agg(
    child_count=("node_id", "count"),
    excluded_count=("index_exclude", "sum"),
    avg_chars=("char_count", "mean"),
    min_chars=("char_count", "min"),
    max_chars=("char_count", "max"),
).reset_index()

summary_path = META_DIR / "corpus_preparation_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved corrected summary:", summary_path)
display(summary_df)

role_summary = child_df.groupby([
    "doc_id",
    "evidence_role",
]).agg(
    count=("node_id", "count"),
    avg_chars=("char_count", "mean"),
).reset_index().sort_values(
    ["doc_id", "count"],
    ascending=[True, False],
)

display(role_summary)

Saved corrected summary: /content/drive/MyDrive/llm/rag_metadata/corpus_preparation_summary.csv


,doc_id,document_type,authority_level,child_count,excluded_count,avg_chars,min_chars,max_chars
0,aha_2022_guideline,clinical_guideline,high,890,0,933.288764,99,6320
1,aha_discharge_packet,patient_education,supportive,84,0,842.285714,87,2437
2,nice_acute_hf,clinical_guideline,high,28,0,419.535714,72,5708
3,nice_chronic_hf,clinical_guideline,high,87,0,286.551724,78,1605


,doc_id,evidence_role,count,avg_chars
1,aha_2022_guideline,supportive_text,318,896.056604
0,aha_2022_guideline,recommendation,259,951.471042
3,aha_2022_guideline,table_or_structured_evidence,234,977.252137
2,aha_2022_guideline,synopsis,79,893.329114
5,aha_discharge_packet,patient_lifestyle_instruction,48,881.750000
4,aha_discharge_packet,patient_instruction,13,620.384615
7,aha_discharge_packet,warning_sign_or_self_check_instruction,12,745.583333
6,aha_discharge_packet,patient_medication_instruction,11,1037.818182
8,nice_acute_hf,recommendation,28,419.535714
9,nice_chronic_hf,recommendation,86,282.430233


In [46]:
# ============================================================
# FIXED CELL 20
# Check that NICE parent chunking is now correct
# ============================================================

nice_parent_check = parent_summary[
    parent_summary["doc_id"].isin(["nice_chronic_hf", "nice_acute_hf"])
][[
    "doc_id",
    "section_id",
    "section_title",
    "page_start",
    "page_end",
    "is_front_matter",
    "chars",
]]

display(nice_parent_check)

print("NICE chronic parent count:", len(nice_parent_check[nice_parent_check["doc_id"] == "nice_chronic_hf"]))
print("NICE acute parent count  :", len(nice_parent_check[nice_parent_check["doc_id"] == "nice_acute_hf"]))

,doc_id,section_id,section_title,page_start,page_end,is_front_matter,chars
86,nice_chronic_hf,front_matter,Front matter / introductory material,1,5,True,7252
87,nice_chronic_hf,1.1,Team working in the management of heart failure,6,8,False,3825
88,nice_chronic_hf,1.2,Diagnosing heart failure,9,11,False,4272
89,nice_chronic_hf,1.3,Giving information to people with heart failure,12,12,False,853
90,nice_chronic_hf,1.4,Treating people with newly diagnosed and preex...,13,15,False,4011
91,nice_chronic_hf,1.5,Treating people with newly diagnosed and preex...,16,16,False,2473
92,nice_chronic_hf,1.6,Treating heart failure in people with chronic ...,17,17,False,510
93,nice_chronic_hf,1.7,Starting and monitoring medication use,18,20,False,4431
94,nice_chronic_hf,1.8,Clinical review,21,21,False,1597
95,nice_chronic_hf,1.9,Other treatments and advice for all types of h...,22,23,False,2660


NICE chronic parent count: 13
NICE acute parent count  : 8


In [47]:
# ============================================================
# FIXED CELL 21
# Check AHA parent headings
# The false headings like "1. Clinical congestion..." should disappear.
# ============================================================

aha_parent_check = parent_summary[
    parent_summary["doc_id"] == "aha_2022_guideline"
][[
    "section_id",
    "section_title",
    "page_start",
    "page_end",
    "is_front_matter",
    "chars",
]]

display(aha_parent_check.head(40))

# Find suspicious AHA headings.
suspicious_aha = aha_parent_check[
    aha_parent_check["section_id"].astype(str).str.match(r"^[0-9]$")
]

print("Suspicious single-number AHA parent headings:", len(suspicious_aha))
display(suspicious_aha)

,section_id,section_title,page_start,page_end,is_front_matter,chars
0,front_matter,Front matter / introductory material,1,4,True,22350
1,1.2,Organization of the Writing Committee,5,5,False,1170
2,1.1,Methodology and Evidence Review,5,5,False,1375
3,1.4,Scope of the Guideline,5,5,False,2389
4,1.5,Class of Recommendation and Level of Evidence,6,6,False,421
5,1.6,Abbreviations,6,6,True,3939
6,2.1,Stages of HF,7,7,False,1846
7,2.2,Classification of HF by Left Ventricular Eject...,7,10,False,13720
8,2.3,Diagnostic Algorithm for Classification of HF ...,11,11,False,4121
9,3.1,Epidemiology of HF,12,12,False,3340


Suspicious single-number AHA parent headings: 0


,section_id,section_title,page_start,page_end,is_front_matter,chars


In [48]:
# ============================================================
# FIXED CELL 22
# Inspect clinically important child chunks after fixes
# ============================================================

def show_sample_children(
    doc_id: Optional[str] = None,
    topic_contains: Optional[str] = None,
    role_contains: Optional[str] = None,
    n: int = 5,
    max_chars: int = 900,
):
    rows = all_child_nodes

    if doc_id:
        rows = [r for r in rows if r["doc_id"] == doc_id]

    if topic_contains:
        rows = [
            r for r in rows
            if any(
                topic_contains.lower() in t.lower()
                for t in r["metadata"].get("topic", [])
            )
        ]

    if role_contains:
        rows = [
            r for r in rows
            if role_contains.lower() in r["metadata"].get("evidence_role", "").lower()
        ]

    rows = [r for r in rows if not r["metadata"].get("index_exclude", False)]

    print(f"Matching child chunks: {len(rows)}")

    for r in rows[:n]:
        m = r["metadata"]

        print("=" * 100)
        print("node_id           :", r["node_id"])
        print("doc_id            :", r["doc_id"])
        print("source_file       :", m["source_file"])
        print("section           :", m["section_id"], "-", m["section_title"])
        print("recommendation_id :", m["recommendation_id"])
        print("evidence_role     :", m["evidence_role"])
        print("topics            :", m["topic"])
        print("pages             :", m["page_start"], "-", m["page_end"])
        print("-" * 100)
        print(r["text"][:max_chars])
        print()


print("\n--- Medication monitoring / renal function / electrolytes ---")
show_sample_children(
    topic_contains="renal_function_electrolytes",
    n=4,
    max_chars=700,
)

print("\n--- Warning signs / self-check instructions ---")
show_sample_children(
    topic_contains="warning_signs",
    n=4,
    max_chars=700,
)

print("\n--- Post-discharge follow-up ---")
show_sample_children(
    topic_contains="post_discharge_follow_up",
    n=4,
    max_chars=700,
)


--- Medication monitoring / renal function / electrolytes ---
Matching child chunks: 95
node_id           : aha_2022_guideline_child_00019_c53fa96f
doc_id            : aha_2022_guideline
source_file       : 1 heidenreich-et-al-2022-2022-aha-acc-hfsa-guideline-for-the-management-of-heart-failure-a-report-of-the-american-college.pdf
section           : 2.2 - Classification of HF by Left Ventricular Ejection Fraction (LVEF)
recommendation_id : None
evidence_role     : table_or_structured_evidence
topics            : ['monitoring', 'renal_function_electrolytes', 'natriuretic_peptides', 'diagnosis_assessment']
pages             : 7 - 10
----------------------------------------------------------------------------------------------------
Stages of HF |Stages|Definition and Criteria| |---|---| |Stage A: At Risk for HF|At risk for HF but without symptoms, structural heart disease, or cardiac biomarkers of stretch or injury (eg, patients with<br>hypertension, atherosclerotic CVD, diabetes, meta

In [49]:
# ============================================================
# FIXED CELL 23
# Final validation checks
# ============================================================

def validate_corpus(
    parent_nodes: List[Dict[str, Any]],
    child_nodes: List[Dict[str, Any]],
) -> None:

    assert len(parent_nodes) > 0, "No parent nodes were created."
    assert len(child_nodes) > 0, "No child nodes were created."

    parent_ids = {p["parent_id"] for p in parent_nodes}
    child_parent_ids = {c["parent_id"] for c in child_nodes}

    missing_parent_refs = child_parent_ids - parent_ids

    assert not missing_parent_refs, (
        f"Some child nodes reference missing parents: "
        f"{list(missing_parent_refs)[:5]}"
    )

    child_ids = [c["node_id"] for c in child_nodes]
    assert len(child_ids) == len(set(child_ids)), "Duplicate child node IDs found."

    required_meta = [
        "source_file",
        "source_title",
        "publisher",
        "document_type",
        "audience",
        "clinical_phase",
        "authority_level",
        "section_id",
        "section_title",
        "evidence_role",
        "topic",
        "page_start",
        "page_end",
        "is_front_matter",
        "index_exclude",
    ]

    for c in child_nodes:
        for key in required_meta:
            assert key in c["metadata"], f"Missing metadata key {key} in {c['node_id']}"

        assert len(c["text"].strip()) >= 50, f"Very short child node: {c['node_id']}"

    # Important thesis-specific checks.
    nice_chronic_parents = [
        p for p in parent_nodes
        if p["doc_id"] == "nice_chronic_hf" and not p["metadata"].get("index_exclude", False)
    ]

    nice_acute_parents = [
        p for p in parent_nodes
        if p["doc_id"] == "nice_acute_hf" and not p["metadata"].get("index_exclude", False)
    ]

    assert len(nice_chronic_parents) > 3, "NICE chronic still has too few parent sections."
    assert len(nice_acute_parents) > 3, "NICE acute still has too few parent sections."

    aha_single_number_sections = [
        p for p in parent_nodes
        if p["doc_id"] == "aha_2022_guideline"
        and re.match(r"^[0-9]$", str(p["metadata"].get("section_id", "")))
    ]

    assert len(aha_single_number_sections) == 0, "AHA still has false single-number parent headings."

    print("Validation passed.")
    print(f"Parent nodes: {len(parent_nodes)}")
    print(f"Child nodes : {len(child_nodes)}")
    print(f"NICE chronic parent sections: {len(nice_chronic_parents)}")
    print(f"NICE acute parent sections  : {len(nice_acute_parents)}")
    print(f"AHA false single-number parent headings: {len(aha_single_number_sections)}")


validate_corpus(all_parent_nodes, all_child_nodes)

Validation passed.
Parent nodes: 149
Child nodes : 1089
NICE chronic parent sections: 12
NICE acute parent sections  : 7
AHA false single-number parent headings: 0


In [50]:
# ============================================================
# FIXED CELL 24
# Save corrected parent lookup for Thesis_4_RAG_implement.ipynb
# ============================================================

parent_lookup = {
    p["parent_id"]: {
        "doc_id": p["doc_id"],
        "text": p["text"],
        "metadata": p["metadata"],
    }
    for p in all_parent_nodes
}

parent_lookup_path = META_DIR / "parent_lookup.json"

with parent_lookup_path.open("w", encoding="utf-8") as f:
    json.dump(parent_lookup, f, ensure_ascii=False, indent=2)

print("Saved corrected parent lookup:", parent_lookup_path)
print("Ready for Thesis_4_RAG_implement.ipynb")

Saved corrected parent lookup: /content/drive/MyDrive/llm/rag_metadata/parent_lookup.json
Ready for Thesis_4_RAG_implement.ipynb


In [51]:
# ============================================================
# FIXED CELL 25
# Final output check
# ============================================================

required_outputs = [
    META_DIR / "document_inventory.csv",
    META_DIR / "parsed_pages.jsonl",
    META_DIR / "parent_nodes.jsonl",
    META_DIR / "child_nodes.jsonl",
    META_DIR / "parent_lookup.json",
    META_DIR / "corpus_preparation_summary.csv",
]

for path in required_outputs:
    print(path, "EXISTS:", path.exists())

/content/drive/MyDrive/llm/rag_metadata/document_inventory.csv EXISTS: True
/content/drive/MyDrive/llm/rag_metadata/parsed_pages.jsonl EXISTS: True
/content/drive/MyDrive/llm/rag_metadata/parent_nodes.jsonl EXISTS: True
/content/drive/MyDrive/llm/rag_metadata/child_nodes.jsonl EXISTS: True
/content/drive/MyDrive/llm/rag_metadata/parent_lookup.json EXISTS: True
/content/drive/MyDrive/llm/rag_metadata/corpus_preparation_summary.csv EXISTS: True
